In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
# from matplot2tikz import save
from tqdm import tqdm

from enc_func import Dec, Enc_state, Enc_t, EncryptionConfig, Mod, Params, Seret_key
from FGH import FGHConfig, LQRConfig, PlantConfig, build_model_data, compute_offline_mats, float_to_object_int, save_offline_mats


In [ ]:
# Select the encryption parameters used by enc_func.py.
ENC_CONFIG = EncryptionConfig(
    p=2**65,
    L=2**44,
    r=19.2,
    N=2**12,
    q_offset=31,
)

# Select the plant discretization and controller parameters used by FGH.py.
FGH_CONFIG = FGHConfig(
    plant=PlantConfig(
        J1=0.01,
        J2=0.01,
        J3=0.01,
        k1=1.37,
        k2=1.37,
        b1=0.007,
        b2=0.007,
        b3=0.007,
        Ts=0.1,
    ),
    lqr=LQRConfig(
        Q_diag=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0),
        R_diag=(1.0,),
    ),
)

# Quantization scales used in this notebook experiment.
r_quant = 100000
s_quant = 100000


In [ ]:
# Build the encryption environment and the FGH model from the selected parameters.
Ts = FGH_CONFIG.plant.Ts
env = Params(ENC_CONFIG)
sk = Seret_key(env)
model = build_model_data(FGH_CONFIG)

# Rebuild the offline matrices from the same FGH configuration and cache them as NumPy data.
offline = compute_offline_mats(env, s_quant, config=FGH_CONFIG, model=model)
save_offline_mats(offline)
print("Saved offline matrices")

F_bar = offline["F_bar"]
G_bar = offline["G_bar"]
H_bar = offline["H_bar"]
Phi_pinv_bar = offline["Phi_pinv_bar"]
T1_all = offline["T1_all"]
T2_all = offline["T2_all"]
V2_all = offline["V2_all"]
S_xi_all = offline["S_xi_all"]
S_v_all = offline["S_v_all"]
Psi_all = offline["Psi_all"]
Sigma_all = offline["Sigma_all"]
Sigma_pinv_all = offline["Sigma_pinv_all"]

# Discrete-time plant and controller matrices come from the same FGH configuration.
A = model.A
B = model.B.reshape(-1)
C = model.C
K = model.K.reshape(-1)


In [ ]:
# Simulation configuration.
iter_count = 10
n_channels = 60
execution_times = []

# Plant and observer initial conditions.
xp0 = np.ones((6, 1), dtype=float)
z_hat0 = np.zeros((24, 1), dtype=float)
z_hat_bar = np.round(z_hat0 * r_quant * s_quant).astype(int)

# Arrays used for plots and diagnostics.
attack_arr = np.zeros(iter_count)
attack_start = iter_count // 2
xp = [xp0]
u = []
y = []
x_hat_list = []
residue = np.zeros((iter_count, n_channels))

# Encrypt the initial observer state for every residual channel.
Z_hat_list = []
b_xi_list = []
for T1_j, T2_j, V2_j in zip(T1_all, T2_all, V2_all):
    Z_hat_j, b_xi_j = Enc_state(z_hat_bar, sk, env, T1_j, T2_j, V2_j)
    Z_hat_list.append(Z_hat_j)
    b_xi_list.append(b_xi_j)

# Recover x_hat(0) from one representative channel.
Z_hat_ref0 = Z_hat_list[0]
X_hat_cipher0 = Mod(Phi_pinv_bar @ Z_hat_ref0, env.q)
x_hat_int0 = Dec(X_hat_cipher0, sk, env)
x_hat_list = [x_hat_int0 / (r_quant * s_quant * s_quant * env.L)]


In [ ]:
channel_terms = [
    (
        H_bar[channel_idx, :].reshape(1, 24),
        S_xi_all[channel_idx],
        S_v_all[channel_idx],
        Psi_all[channel_idx],
        Sigma_all[channel_idx],
        Sigma_pinv_all[channel_idx],
    )
    for channel_idx in range(n_channels)
]
B_col = B.reshape(-1, 1)

for k in tqdm(range(iter_count)):
    t_start = time.perf_counter()

    # Compute the current plant output and control input.
    y_k = C @ xp[-1]
    y.append(y_k)

    u_k = (K @ xp[-1]).item()
    u.append(u_k)

    # Stack the control input and sensor outputs into v = [u; y].
    v = np.vstack([np.array([[u_k]]), y_k])

    # Inject the notebook attack profile into sensor 3.
    if k >= attack_start:
        crit = (k - attack_start) / iter_count
        if crit < 0.1:
            attack = 1.0
        elif 0.35 <= crit < 0.4:
            attack = -1.0
        else:
            attack = 0.0
    else:
        attack = 0.0

    attack_arr[k] = attack
    v[3, 0] += attack

    # Quantize the plaintext input/output vector before encryption.
    v_bar = float_to_object_int(v * r_quant)

    for channel_idx, (H_j, S_xi_j, S_v_j, Psi_j, Sigma_j, Sigma_pinv_j) in enumerate(channel_terms):
        Z_hat_j = Z_hat_list[channel_idx]
        b_xi_j = b_xi_list[channel_idx]

        # Recover the residual scalar from the first ciphertext column.
        R_j = Mod(H_j @ Z_hat_j, env.q)
        bbr_j = Mod(R_j[0, 0], env.q)
        r_bar_j = Mod(bbr_j * env.invL, env.q)
        residue[k, channel_idx] = float(r_bar_j) / (r_quant * s_quant * s_quant)

        V_j, b_v_j = Enc_t(v_bar, sk, b_xi_j, Sigma_pinv_j, Sigma_j, Psi_j, env)

        # Update the encrypted observer state and the dynamic masking state.
        Z_hat_list[channel_idx] = Mod(F_bar @ Z_hat_j + G_bar @ V_j, env.q)
        b_xi_list[channel_idx] = Mod(S_xi_j @ b_xi_j + S_v_j @ b_v_j, env.q)

    # Decrypt one representative observer state to recover the estimated plant state.
    Z_hat_ref = Z_hat_list[0]
    X_hat_cipher = Mod(Phi_pinv_bar @ Z_hat_ref, env.q)
    x_hat_int = Dec(X_hat_cipher, sk, env)
    x_hat_list.append(x_hat_int / (r_quant * s_quant * s_quant * env.L))

    # Advance the plaintext plant dynamics.
    xp.append(A @ xp[-1] + B_col * u_k)

    execution_times.append(time.perf_counter() - t_start)

execution_times = np.array(execution_times)
print(f"Iteration time min  : {execution_times.min():.6f} s")
print(f"Iteration time max  : {execution_times.max():.6f} s")
print(f"Iteration time mean : {execution_times.mean():.6f} s")


In [ ]:
# Assemble arrays for plotting.
t_u = np.arange(len(u))
xp_arr = np.hstack(xp)
x_hat_arr = np.hstack(x_hat_list)
t_k = np.arange(iter_count)

# Plot the estimation error, injected attack, and residual infinity norm.
e_arr = xp_arr[:, 1:] - x_hat_arr[:, 1:]
err_inf = np.max(np.abs(e_arr), axis=0)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

axes[0].plot(t_k, err_inf, linestyle='-')
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1)
axes[0].grid(True)
axes[0].set_ylabel(r"$\|x_p(k)-\hat{x}(k)\|_\infty$")
axes[0].set_title("State estimation error")

axes[1].plot(t_u, attack_arr)
axes[1].grid(True)
axes[1].set_ylabel("attack")
axes[1].set_title("Injected Attack on sensor 3")

r_inf = np.max(np.abs(residue), axis=1)
axes[2].plot(t_u, r_inf, label=r"$\|r\|_\infty$", linestyle='-')
axes[2].axhline(0.1, color='red', linestyle='--', linewidth=1)
axes[2].grid(True)
axes[2].set_xlabel("time")
axes[2].set_ylabel(r"$\|r\|_\infty$")
axes[2].set_title("Residual infinity norm")

# save("myplot.tex")
fig.tight_layout()
plt.show()
